# NanoTSE Colab A100 training

Trains the 27M-param NanoTSE on Colab A100 reading data from Google Drive.

**Prerequisites (do these once):**
1. Upload `data/v2/` from your local 3060 box to Drive at `MyDrive/nanotse/data/v2/`.  
   Use [rclone](https://rclone.org/) or Drive sync. Total ~62 GB (28 GB audio + 35 GB faces).
2. Make sure this notebook is in Colab with A100 GPU selected (Runtime -> Change runtime type).

**Each session:** run the cells in order. Training auto-saves checkpoints back to Drive so disconnects are recoverable.

## 1. Mount Google Drive + verify data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
DRIVE_ROOT = '/content/drive/MyDrive/nanotse'
DATA_ROOT = f'{DRIVE_ROOT}/data/v2'
assert os.path.exists(f'{DATA_ROOT}/manifest.json'), f'manifest.json not found at {DATA_ROOT}'
m = json.load(open(f'{DATA_ROOT}/manifest.json'))
print(f'manifest ok: {len(m["train"])} train clips, {len(m["val"])} val clips')
print(f'speakers: train={len({r["speaker_id"] for r in m["train"]})}, val={len({r["speaker_id"] for r in m["val"]})}')
!df -h /content/drive | head -3

## 2. Clone the repo + install dependencies

Replace `<git-remote-url>` with your repo URL, or use `repo.tar.gz` if you uploaded one to Drive.

In [ ]:
%cd /content
# Option A: git clone (replace URL)
# !git clone https://github.com/<your-username>/nanotse.git

# Option B: extract from Drive tarball (if you used scripts/prepare_for_colab.sh)
import os
if not os.path.exists('/content/nanotse'):
    !mkdir -p /content/nanotse && tar -xzf /content/drive/MyDrive/nanotse/repo.tar.gz -C /content/nanotse

%cd /content/nanotse
!pip install -q --upgrade pip
!pip install -q pydantic pyyaml soundfile opencv-python-headless
# torch is preinstalled on Colab A100 images; if not:
# !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
print('deps ready')
!python -c 'import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")'

## 3. Patch the config to point at Drive data + Drive run dir

Two changes from `configs/colab_a100_v1.yaml`:
1. `data.root` -> Drive-mounted absolute path (already set in the YAML)
2. The training script writes to `runs/<run-name>/` relative to repo root. We symlink that to Drive so checkpoints survive session disconnects.

In [ ]:
import os
RUN_NAME = 'colab_a100_v1'
DRIVE_RUNS = '/content/drive/MyDrive/nanotse/runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
os.makedirs(f'{DRIVE_RUNS}/{RUN_NAME}', exist_ok=True)

# Symlink repo's runs/<run_name>/ to Drive so checkpoints persist
local_run = f'/content/nanotse/runs/{RUN_NAME}'
os.makedirs('/content/nanotse/runs', exist_ok=True)
if os.path.lexists(local_run):
    !rm -rf {local_run}
!ln -s {DRIVE_RUNS}/{RUN_NAME} {local_run}
print(f'run dir: {local_run} -> {DRIVE_RUNS}/{RUN_NAME}')
!ls -la {local_run}

## 4. Fire training (background) + tail the log

Runs `scripts/train.py` with the colab config. Pipes output to a log file in the symlinked run dir so it's persisted to Drive.

In [ ]:
%cd /content/nanotse
# Detect whether we have an existing checkpoint to resume from
import os
resume_arg = ''
latest = f'/content/nanotse/runs/{RUN_NAME}/latest.pt'
if os.path.exists(latest):
    resume_arg = f'--resume {RUN_NAME}'
    print(f'Resuming from {latest}')
else:
    print('Fresh start')

log_path = f'/content/nanotse/runs/{RUN_NAME}/stdout.log'
cmd = f'cd /content/nanotse && python -u scripts/train.py --config configs/colab_a100_v1.yaml --run-name {RUN_NAME} {resume_arg}'
print('launching:', cmd)
# Run in foreground so the cell streams output
!{cmd} 2>&1 | tee -a {log_path}

## 5. Monitor progress (run from another cell while training runs)

Cell above blocks until training finishes. If you want to monitor without blocking, change `!{cmd}` to `!nohup {cmd} > /dev/null 2>&1 &` and use this cell.

In [ ]:
import json, statistics
metrics_path = f'/content/nanotse/runs/{RUN_NAME}/metrics.jsonl'
if not os.path.exists(metrics_path):
    print('no metrics yet')
else:
    rows = [json.loads(l) for l in open(metrics_path)]
    train = [r for r in rows if 'si_snr_db' in r and 'val_n' not in r]
    val = [r for r in rows if 'val_sdri_db' in r]
    last = train[-1] if train else None
    if last:
        print(f'step {last["step"]}  wall={last["t"]/60:.1f}min')
        print(f'train SI-SNR last_5={statistics.fmean(r["si_snr_db"] for r in train[-5:]):+.2f}  best={max(r["si_snr_db"] for r in train):+.2f} dB')
    if val:
        print(f'val SDRi: {" -> ".join(f"{v["val_sdri_db"]:+.2f}" for v in val)}  best={max(v["val_sdri_db"] for v in val):+.2f} dB')

## 6. Resume after disconnect

If Colab disconnects and you reconnect later, just re-run cells 1-4 in order. Cell 4 auto-detects `latest.pt` in the Drive-backed run dir and passes `--resume <run-name>`.